## Problem 4 - The effect of Article VIII

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("treaty.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df[["restrict", "art8", "imf_ccode"]].describe())

(4362, 48)
imf_ccode                 int64
year                      int64
art8                      int64
restrict                  int64
flexible                  int64
gnpcap                  float64
regnorm                 float64
gdpgrow                 float64
resgdp                  float64
bopgdp                  float64
useimfcr                  int64
surveil                   int64
univers                 float64
resvol                  float64
totvol                  float64
tradedep                float64
lastrest                  int64
lastrest2                 int64
lastrest3                 int64
military                float64
termlim                 float64
parli                   float64
shift_left              float64
leftshift_inc           float64
rightshift_inc          float64
leftshift_exc           float64
rightshift_exc          float64
leftshift_inc_leg       float64
rightshift_inc_leg      float64
leftshift_exc_leg       float64
rightshift_exc_leg      float

The dataset contains 4,362 country-year observations across 48 variables. The outcome `restrict` and treatment `art8` are both binary, with 54% of observations recording a currency restriction and 41% having signed Article VIII. After dropping rows with missing covariate values the analysis sample is smaller. Which I will report after part a

**Part (a)**

In [2]:
# Covs given in question 
covs = ["shift_left", "flexible", "regnorm", "gdpgrow",
        "resgdp", "bopgdp", "useimfcr", "surveil", "univers",
        "resvol", "totvol", "tradedep", "military", "termlim",
        "parli", "lastrest", "lastrest2", "lastrest3"]

cols = ["restrict", "art8", "imf_ccode"] + covs
df_model = df[cols].dropna()
print(f"Rows after dropping NAs: {len(df_model)} (dropped {len(df) - len(df_model)})")

X = sm.add_constant(df_model[["art8"] + covs])
y = df_model["restrict"]

logit_model = sm.Logit(y, X)
logit_result = logit_model.fit()
print(logit_result.summary2())

art8_coef = logit_result.params["art8"]
art8_se = logit_result.bse["art8"]
art8_pval = logit_result.pvalues["art8"]
print(f"\nart8 coefficient: {art8_coef:.4f}")
print(f"art8 SE: {art8_se:.4f}")
print(f"art8 p-value: {art8_pval:.4f}")
print(f"art8 odds ratio: {np.exp(art8_coef):.4f}")

Rows after dropping NAs: 4362 (dropped 0)
Optimization terminated successfully.
         Current function value: 0.259201
         Iterations 8
                         Results: Logit
Model:              Logit            Method:           MLE      
Dependent Variable: restrict         Pseudo R-squared: 0.624    
Date:               2026-05-04 14:17 AIC:              2301.2731
No. Observations:   4362             BIC:              2428.8868
Df Model:           19               Log-Likelihood:   -1130.6  
Df Residuals:       4342             LL-Null:          -3006.8  
Converged:          1.0000           LLR p-value:      0.0000   
No. Iterations:     8.0000           Scale:            1.0000   
-----------------------------------------------------------------
             Coef.   Std.Err.     z      P>|z|    [0.025   0.975]
-----------------------------------------------------------------
const        0.2730    0.6013    0.4540  0.6498  -0.9055   1.4515
art8        -1.6956    0.1621  -

In [3]:
print(np.exp(art8_coef))

0.18349751860998667


The logistic regression gets `art8` coefficient of -1.696 (with SE = 0.162, p < 0.001),which corresponds to an odds ratio of 0.184. Intuitively, after adjusting for all covariates, countries that have signed Article VIII have roughly 82% lower odds of imposing a currency restriction in a given year compared to non-signatories. It is important to note that this should not be interpreted as a causal effect. Countries self-select into signing Article VIII, so those that sign are likely already more committed to open capital markets, so the association may simply reflect this selection rather than any effect of the treaty itself. (Treatment is not random!!) Without a credible source of exogenous variation in treaty adoption, the strong negative association, no matter how statistically significant it is, cannot be given a causal interpretation.

**Part (b)**

In [5]:
logit_robust = logit_model.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_model["imf_ccode"]}
)
print(logit_robust.summary2())

art8_coef_r = logit_robust.params["art8"]
art8_se_r = logit_robust.bse["art8"]
art8_pval_r = logit_robust.pvalues["art8"]
print(f"\nart8 coefficient: {art8_coef_r:.4f}")
print(f"art8 cluster-robust SE: {art8_se_r:.4f}")
print(f"art8 p-value: {art8_pval_r:.4f}")
print(f"art8 odds ratio: {np.exp(art8_coef_r):.4f}")

print(f"\nSE comparison:")
print(f"Naive SE: {art8_se:.4f}")
print(f"Cluster-robust SE: {art8_se_r:.4f}")

Optimization terminated successfully.
         Current function value: 0.259201
         Iterations 8
                         Results: Logit
Model:              Logit            Method:           MLE      
Dependent Variable: restrict         Pseudo R-squared: 0.624    
Date:               2026-05-04 14:23 AIC:              2301.2731
No. Observations:   4362             BIC:              2428.8868
Df Model:           19               Log-Likelihood:   -1130.6  
Df Residuals:       4342             LL-Null:          -3006.8  
Converged:          1.0000           LLR p-value:      0.0000   
No. Iterations:     8.0000           Scale:            1.0000   
-----------------------------------------------------------------
             Coef.   Std.Err.     z      P>|z|    [0.025   0.975]
-----------------------------------------------------------------
const        0.2730    0.8336    0.3275  0.7433  -1.3608   1.9068
art8        -1.6956    0.2479   -6.8410  0.0000  -2.1813  -1.2098
shift_le

Refitting the model with standard errors clustered by country (`imf_ccode`), the `art8` coefficient is unchanged, but the standard error increases from 0.162 to 0.248. This is expected as country-year observations within the same country are correlated over time, so the naive SE understates uncertainty by treating them as independent. But despite the larger SE, my conclusion does not change. The `art8` coefficient remains highly significant (p < 0.001), and the negative association between signing Article VIII and imposing currency restrictions is robust to clustering.

**Part (c)**

I disagree with this hypothetical suggestion. Ridge regression is most appropriate when the goal is prediction, where shrinkage reduces variance at the cost of introducing bias. Here I'd say the goal is more inference. Applying a ridge penalty would bias the `art8` coefficient toward zero, making it difficult to assess its true magnitude, and the resulting standard errors would no longer support valid hypothesis tests. Any penalized regression method, including Lasso aswell (L1), shares this problem.Regularization is the wrong tool when the goal is unbiased inference on a specific coefficient rather than prediction.